In [1]:
!pip install -U unsloth trl datasets

  Using cached trl-1.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)


In [2]:
"""
Notebook-ready Doggolingo QLoRA training code.

Kaggle/Colab install cell:
    !pip install -U unsloth trl datasets

If Kaggle stores your dataset under /kaggle/input, run:
    !find /kaggle -maxdepth 5 -type f | grep doggolingo

Then update dataset_path below.
"""

from pathlib import Path
import inspect

from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTConfig, SFTTrainer


# Basic config
model_name = "unsloth/Qwen3-1.7B"
dataset_path = "/kaggle/input/datasets/elenali123/training/doggolingo_sft.jsonl"
output_dir = "/kaggle/working/doggolingo-qlora-adapter"

max_seq_length = 1024
seed = 3407

# LoRA / QLoRA config
r = 16
lora_alpha = 16
lora_dropout = 0.0
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# Training config
num_train_epochs = 3
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
learning_rate = 2e-4
warmup_steps = 10
logging_steps = 5
save_steps = 100
eval_steps = 100
eval_split_ratio = 0.05
packing = False


dataset_path = Path(dataset_path)
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found: {dataset_path}")

dataset = load_dataset("json", data_files=str(dataset_path), split="train")
if "messages" not in dataset.column_names:
    raise ValueError("Dataset must contain a `messages` column.")

if len(dataset) >= 20:
    split = dataset.train_test_split(test_size=eval_split_ratio, seed=seed)
    train_dataset = split["train"]
    eval_dataset = split["test"]
else:
    train_dataset = dataset
    eval_dataset = None

print(f"Training examples: {len(train_dataset)}")
if eval_dataset is not None:
    print(f"Validation examples: {len(eval_dataset)}")
else:
    print("Validation split skipped because dataset is too small.")









🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Generating train split: 0 examples [00:00, ? examples/s]

Training examples: 475
Validation examples: 25


In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=seed,
)

config_kwargs = {
    "output_dir": output_dir,
    "max_length": max_seq_length,
    "num_train_epochs": num_train_epochs,
    "per_device_train_batch_size": per_device_train_batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "learning_rate": learning_rate,
    "warmup_steps": warmup_steps,
    "logging_steps": logging_steps,
    "save_steps": save_steps,
    "save_total_limit": 2,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": seed,
    "packing": packing,
    "report_to": "none",
    "assistant_only_loss": False,
}

signature = inspect.signature(SFTConfig)
if eval_dataset is not None:
    if "eval_strategy" in signature.parameters:
        config_kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in signature.parameters:
        config_kwargs["evaluation_strategy"] = "steps"
    config_kwargs["eval_steps"] = eval_steps


if "fp16" in signature.parameters:
    config_kwargs["fp16"] = not is_bfloat16_supported()
if "bf16" in signature.parameters:
    config_kwargs["bf16"] = is_bfloat16_supported()

# Force Qwen EOS before creating SFTConfig
qwen_eos = tokenizer.eos_token  # should be <|im_end|>

config_kwargs["eos_token"] = qwen_eos

training_args = SFTConfig(**config_kwargs)

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
def formatting_func(examples):
    messages = examples["messages"]
    if messages and isinstance(messages[0], dict):
        conversations = [messages]
    else:
        conversations = messages

    return [
        tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )
        for conversation in conversations
    ]


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_func,
)

trainer.train()

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Saved LoRA adapter to: {output_dir}")


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/475 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/25 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 475 | Num Epochs = 3 | Total steps = 180
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,1.744294,1.972911
180,1.516163,1.958317


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/doggolingo-qlora-adapter/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/doggolingo-qlora-adapter/checkpoint-180/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/doggolingo-qlora-adapter/tokenizer_config.json.


Saved LoRA adapter to: /kaggle/working/doggolingo-qlora-adapter


In [5]:
from unsloth import FastLanguageModel
import re

FastLanguageModel.for_inference(model)

def ask_doggo(prompt, max_new_tokens=120):
  messages = [
      {"role": "user", "content": prompt}
  ]

  encoded = tokenizer.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True,
  )

  encoded = {k: v.to("cuda") for k, v in encoded.items()}

  outputs = model.generate(
      **encoded,
      max_new_tokens=max_new_tokens,
      temperature=0.8,
      top_p=0.9,
      do_sample=True,
      repetition_penalty=1.08,
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.eos_token_id,
  )

  input_length = encoded["input_ids"].shape[-1]

  response = tokenizer.decode(
      outputs[0][input_length:],
      skip_special_tokens=True,
  )

  response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
  return response.strip()

In [6]:
ask_doggo("What do you think is happening when I leave the house?")

Both `max_new_tokens` (=120) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


"It's a whole world of sniff-sounds, doggo tail wag possibilities, and one brave little foot step away from the familiar floor."